In [0]:
import datetime
import json
from pyspark.sql.functions import lit, current_timestamp


In [0]:
spark.sql("USE CATALOG db_dataclassdev")
spark.sql("USE bronze")


In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text('fileName', 'item')
dbutils.widgets.text('dateHierarchy', '2025/06/25')

In [0]:
fileName = dbutils.widgets.get('fileName')
dateHierarchy = dbutils.widgets.get('dateHierarchy')
fileFullPath = f'abfss://bronze@strdatabrickssadls.dfs.core.windows.net/{fileName}/{dateHierarchy}/{fileName}.csv'
deltaBronzePath = f'abfss://bronze@strdatabrickssadls.dfs.core.windows.net/deltaTables/{fileName}'

In [0]:
try:
    # Read source CSV
    df = spark.read.format('csv').option('header', 'true').load(fileFullPath)
    rowCount = df.count()

    # Add ETL metadata columns
    current_utc = datetime.datetime.utcnow()
    df = (
        df.withColumn("etl_record_created_date", lit(current_utc))
          .withColumn("etl_record_modified_date", current_timestamp())
    )

    # Write to Delta
    df.write.format('delta').mode('overwrite').save(deltaBronzePath)

    # Register as Delta table
    spark.sql(f"CREATE TABLE IF NOT EXISTS {fileName} USING DELTA LOCATION '{deltaBronzePath}'")

    # Audit
    audit_info = {
        "fileName": fileName,
        "rowCount": rowCount,
        "status": "Succeeded",
        "destinationPath": deltaBronzePath,
        "timestamp": str(current_utc)
    }

except Exception as e:
    # On failure
    audit_info = {
        "fileName": fileName,
        "status": "Failed",
        "errorMessage": str(e),
        "timestamp": str(datetime.datetime.utcnow())
    }

# Send audit info to ADF
dbutils.notebook.exit(json.dumps(audit_info))

